# NLP Libraries Practice: spaCy, NLTK, TextBlob, Gensim, Transformers

This notebook uses a Hugging Face dataset (`ag_news`) and demonstrates common NLP workflows across popular libraries.

## 1) Install dependencies (run once)

In [ ]:
%pip install -q datasets spacy nltk textblob gensim scikit-learn transformers torch
!python -m spacy download en_core_web_sm

## 2) Load a dataset from Hugging Face

In [ ]:
from datasets import load_dataset

dataset = load_dataset("ag_news")
dataset

In [ ]:
label_names = dataset["train"].features["label"].names
label_names

In [ ]:
# Keep a smaller sample for fast experimentation
train_small = dataset["train"].shuffle(seed=42).select(range(2000))
test_small = dataset["test"].shuffle(seed=42).select(range(500))

train_small[0]

## 3) spaCy: tokenization, POS, NER

In [ ]:
import spacy

nlp = spacy.load("en_core_web_sm")
sample_text = train_small[0]["text"]
doc = nlp(sample_text)

print("TEXT:", sample_text)
print("\nTOKENS + POS:")
for token in doc[:20]:
    print(f"{token.text:15} {token.pos_:8} {token.dep_}")

print("\nENTITIES:")
for ent in doc.ents:
    print(ent.text, ent.label_)

## 4) NLTK: preprocessing and features

In [ ]:
import nltk

nltk.download("punkt", quiet=True)
nltk.download("stopwords", quiet=True)
nltk.download("averaged_perceptron_tagger", quiet=True)
nltk.download("vader_lexicon", quiet=True)

In [ ]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.sentiment import SentimentIntensityAnalyzer

tokens = word_tokenize(sample_text.lower())
stops = set(stopwords.words("english"))
clean_tokens = [t for t in tokens if t.isalpha() and t not in stops]

stemmer = PorterStemmer()
stemmed = [stemmer.stem(t) for t in clean_tokens[:20]]

print("Clean tokens:", clean_tokens[:20])
print("Stemmed:", stemmed)

sia = SentimentIntensityAnalyzer()
print("VADER sentiment:", sia.polarity_scores(sample_text))

## 5) TextBlob: quick sentiment and noun phrases

In [ ]:
from textblob import TextBlob

blob = TextBlob(sample_text)
print("Sentiment (polarity, subjectivity):", blob.sentiment)
print("Noun phrases:", list(blob.noun_phrases)[:10])

## 6) Gensim: topic modeling (LDA)

In [ ]:
from gensim import corpora
from gensim.models import LdaModel
from nltk.tokenize import word_tokenize

docs = [word_tokenize(x["text"].lower()) for x in train_small.select(range(300))]
docs = [[w for w in d if w.isalpha() and w not in stops] for d in docs]

dictionary = corpora.Dictionary(docs)
corpus = [dictionary.doc2bow(d) for d in docs]

lda = LdaModel(corpus=corpus, id2word=dictionary, num_topics=4, random_state=42, passes=5)
for i, topic in lda.print_topics(num_words=8):
    print(f"Topic {i}: {topic}")

## 7) Hugging Face Transformers: zero/few-shot sentiment classification

In [ ]:
from transformers import pipeline

sentiment_pipe = pipeline("sentiment-analysis")

examples = [
    "The earnings report exceeded expectations and guidance was raised.",
    "Revenue dropped significantly and the outlook was cut for next quarter.",
]

preds = sentiment_pipe(examples)
for text, pred in zip(examples, preds):
    print(text)
    print(pred)
    print()

## 8) Transformers + AG News: quick text classification evaluation

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

model_id = "cardiffnlp/twitter-roberta-base-sentiment-latest"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSequenceClassification.from_pretrained(model_id)

id2label = model.config.id2label

def predict_sentiment(texts):
    inputs = tokenizer(texts, padding=True, truncation=True, return_tensors="pt")
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = torch.softmax(logits, dim=-1)
    pred_ids = probs.argmax(dim=-1).tolist()
    return [(id2label[i], float(probs[j, i])) for j, i in enumerate(pred_ids)]

batch_texts = [test_small[i]["text"] for i in range(8)]
for txt, pred in zip(batch_texts, predict_sentiment(batch_texts)):
    print(pred, "->", txt[:100], "...")

## 9) Optional extension ideas

1. Replace `ag_news` with `imdb` (binary sentiment) and fine-tune a classifier.
2. Try spaCy custom NER training on a small labeled set.
3. Compare NLTK vs spaCy tokenization outputs on finance text.
4. Use Gensim `Word2Vec` and inspect nearest terms for finance keywords.